# Dataset Import

Connect the Google Drive to the notebook

In [3]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
os.chdir("/content/drive/MyDrive/dataset")

In [5]:
!pwd

/content/drive/.shortcut-targets-by-id/1NXU303Jgc2_IKKfoQvUhUL3zarWcR-bv/dataset


In [6]:
!ls

 Clothing1M  'Hotel ID'   Hotels-50k   Receipts  'selfie dataset'   Weapon


## Food 101

Jiawei Sun

**Tutorial:** https://datasets.activeloop.ai/docs/ml/datasets/food-101-dataset/

In [ ]:
!pip install deeplake

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 586.6/586.6 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 20.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 65.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 65.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 90.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 18.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8

In [ ]:
import deeplake
food_101 = deeplake.load('hub://activeloop/food-101-dataset-train')

Opening dataset in read-only mode as you don't have write permissions.


/

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/activeloop/food-101-dataset-train



|

hub://activeloop/food-101-dataset-train loaded successfully.



In [ ]:
dataloader = food_101.pytorch(num_workers=0, batch_size=4, shuffle=False)

/usr/local/lib/python3.10/dist-packages/deeplake/integrations/pytorch/common.py:137: UserWarning: Decode method for tensors ['images'] is defaulting to numpy. Please consider specifying a decode_method in .pytorch() that maximizes the data preprocessing speed based on your transformation.
  warnings.warn(


## Weapon

Jiawei Sun

### Pistol Classification

In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [19]:
pistol_classification = './Weapon/Pistol_classification'

In [28]:
# Initialize lists to store file paths and labels
pistol_class_img_path = []
pistol_class_label = []

# Loop through each subfolder in the base directory
for folder_name in os.listdir(pistol_classification):
    folder_path = os.path.join(pistol_classification, folder_name)
    if os.path.isdir(folder_path):
        for image_filename in os.listdir(folder_path):
            # Construct the full path to the image file
            image_path = os.path.join(folder_path, image_filename)
            # Append the image path and label to the lists
            pistol_class_img_path.append(image_path)
            pistol_class_label.append(folder_name)

In [33]:
# Convert labels to a numerical format
le = LabelEncoder()
numerical_labels = le.fit_transform(pistol_class_label)

# Split the dataset
train_images, val_images, train_labels, val_labels = train_test_split(
    pistol_class_img_path,
    numerical_labels,
    test_size=0.2,  # 20% for validation
    stratify=numerical_labels,  # Stratify split according to the labels
    random_state=42  # Seed for reproducibility
)

# Now you can create your ImageDataGenerators for training and validation
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

# Model training

## Weapon

Jiawei Sun

### Pistol Classfication

In [21]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import load_img, img_to_array

In [36]:
def load_and_preprocess_image(image_path):
    img = load_img(image_path, target_size=(224, 224))  # Resize image
    img = img_to_array(img)
    img /= 255.0  # Normalize image
    return img

In [37]:
# Define a simple CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(np.unique(numerical_labels)), activation='softmax')  # Number of classes
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
# Train the model using a custom data generator
def custom_data_generator(pistol_class_img_path, pistol_class_label, batch_size):
    while True:
        batch_paths = np.random.choice(a=pistol_class_img_path, size=batch_size)
        batch_input = []
        batch_output = []

        for input_path in batch_paths:
            input = load_and_preprocess_image(input_path)
            output = pistol_class_label[pistol_class_img_path.index(input_path)]

            batch_input.append(input)
            batch_output.append(output)

        batch_x = np.stack(batch_input, axis=0)  # Use np.stack instead of np.array
        batch_y = np.array(batch_output)

        yield batch_x, batch_y

# Training the model with a custom generator
batch_size = 32
train_generator = custom_data_generator(train_images, train_labels, batch_size)
val_generator = custom_data_generator(val_images, val_labels, batch_size)

steps_per_epoch = len(train_images) // batch_size
validation_steps = len(val_images) // batch_size

model.fit(train_generator, steps_per_epoch=steps_per_epoch, epochs=10, validation_data=val_generator, validation_steps=validation_steps)

Epoch 1/10
113/246 [============>.................] - ETA: 24:21 - loss: 4.3526 - accuracy: 0.0985